In [ ]:
from sklearn.datasets import load_digits

import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from models import LogisticRegressionClassifier
from core import Pipeline, GridSearchCV
from preprocessing import train_test_split, normalize
import visuals as vis

In [ ]:
digits = load_digits()
X, Y = digits.data, digits.target

vis.style("darkgrid")
vis.plot_class_distribution(digits.target, inverse_transform=lambda i: digits.target_names[i], figsize=(10, 4), chart="bars")

X_train, X_test, Y_train, Y_test = train_test_split(X, Y)


In [ ]:
pipe = Pipeline([
    ("scale" , normalize.z_score()),
    ("lr", LogisticRegressionClassifier())
])

grid = GridSearchCV(
    pipe,
    param_grid={
        "lr@C": [1000, 100, 10, 1, 0.1, 0.01, 0.001],
        "lr@learning_rate": [2, 1, 0.5, 0.25, 0.12, 0.06]
    },
    cv=3,
    scoring="f1"   
)

In [ ]:
grid.fit(X_train, Y_train)

In [ ]:
print(f"Best F1 score: {round(grid.best_score_, 4)}")
vis.plot_confusion_matrices(
    grid.score(X_test, Y_test, scores=["confusion_matrix"], onlyvalues=True),
    digits.target_names,
    titling=lambda _: "Best Estimator Confusion Matrix"
)

In [ ]:
from models import CART_DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier
from time import perf_counter

dt  = CART_DecisionTreeClassifier(max_depth=20, criterion="gini")
clf =      DecisionTreeClassifier(max_depth=20, criterion="gini")
z_score = normalize.z_score()

X_tr, X_ts, Y_tr, Y_ts = train_test_split(X, Y)

z_score.fit_transform(X_tr)
z_score.transform(X_ts)

t1 = perf_counter()
dt.fit(X_tr, Y_tr)
t2 = perf_counter()
clf.fit(X_tr, Y_tr)
t3 = perf_counter()

print("my implamentation:")
print(f"  -  train:    {round(dt.score(X_tr, Y_tr, onlyvalues=True)[0], 4)}")
print(f"  -  test:     {round(dt.score(X_ts, Y_ts, onlyvalues=True)[0], 4)}")
print(f"  -  fit time: {round(t2-t1, 4)}")

print("sklearn implamentation:")
print(f"  -  train:    {round(clf.score(X_tr, Y_tr), 4)}")
print(f"  -  test:     {round(clf.score(X_ts, Y_ts), 4)}")
print(f"  -  fit time: {round(t3-t2, 4)}")